# 第五阶段：RF 与 XGBoost 的 SHAP 概率解释

本 Notebook 展示 `src.modeling.explainability` 生成的真实 SHAP 结果。SHAP 值解释严重逾期概率，而非原始 margin 或 log-odds。

仓库版保留全局汇总，代表客户的逐行解释输出不保存在版本库；本地具备数据产物时可运行以下单元查看。

In [1]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display, Image
from src.modeling.explainability import select_validation_cases

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
summary = json.loads((ROOT / 'reports' / 'stage5_summary.json').read_text(encoding='utf-8'))
print('SHAP 最大概率加总误差：', summary['shap_additivity_max_error'])

SHAP 最大概率加总误差： {'Random Forest': 1.7004851526891684e-08, 'XGBoost': 3.4031125861755385e-07}


In [ ]:
global_shap = pd.read_csv(ROOT / 'reports' / 'stage5_tables' / 'SHAP全局.csv')
cases = pd.read_csv(ROOT / 'reports' / 'stage5_tables' / '解释客户.csv')
local_shap = pd.read_csv(ROOT / 'reports' / 'stage5_tables' / 'SHAP个案.csv')
display(global_shap.groupby('model', group_keys=False).head(10))
display(cases[['case_type']].drop_duplicates())
top_local = local_shap.groupby(['model', 'case_type'], group_keys=False).apply(lambda x: x.reindex(x['shap_probability_contribution'].abs().sort_values(ascending=False).index).head(5), include_groups=False)
display(top_local[['model', 'case_type', 'feature_cn', 'shap_probability_contribution', 'predicted_pd']])

In [3]:
for filename in ['rf_shap_global_bar.png', 'rf_shap_beeswarm.png', 'xgboost_shap_global_bar.png', 'xgboost_shap_beeswarm.png']:
    display(Image(filename=str(ROOT / 'reports' / 'figures' / 'stage5' / filename)))

<IPython.core.display.Image object>
<IPython.core.display.Image object>
<IPython.core.display.Image object>
<IPython.core.display.Image object>


局部解释客户来自验证集的低风险、中位风险、高风险和两树模型分歧最大样本。选例不使用真实标签，不代表因果关系或真实授信结论。